# 🧩 Chapter 7: Unsupervised Learning
**Referensi Buku:** *Practical Statistics for Data Scientists* (Peter Bruce, Andrew Bruce, Peter Gedeck)

---
> **Topik:** Principal Components Analysis (PCA), K-Means Clustering, Hierarchical Clustering, dan Scaling
---
## 1. Pendahuluan
Dalam *Unsupervised Learning*, kita tidak memiliki label target (Y). Tujuannya adalah untuk mengeksplorasi data, menemukan pola tersembunyi, mengelompokkan pelanggan, atau mengurangi dimensi fitur untuk mempercepat proses komputasi.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

%matplotlib inline
np.random.seed(42)

## 2. Principal Components Analysis (PCA)
PCA sering digunakan untuk visualisasi data berdimensi tinggi atau untuk mengurangi *noise*. Di sini kita akan mereduksi dataset yang memiliki banyak fitur ke dalam 2 komponen utama saja agar bisa digambar pada grafik 2D.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA

# Menggunakan dataset Kanker Payudara (memiliki 30 fitur numerik)
cancer = load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target # Label ini hanya kita pakai untuk mewarnai plot, PCA tidak melihatnya

# Wajib melakukan standarisasi (Scaling) sebelum PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cancer)

# Reduksi dari 30 dimensi menjadi 2 dimensi saja
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Bentuk data sebelum PCA: {X_scaled.shape}")
print(f"Bentuk data setelah PCA: {X_pca.shape}")

# Visualisasi 2 Komponen Utama
plt.figure(figsize=(8, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_cancer, cmap='Spectral', alpha=0.7, edgecolors='k')
plt.title('Hasil PCA: Proyeksi 30 Fitur ke 2 Komponen Utama')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(handles=scatter.legend_elements()[0], labels=['Malignant', 'Benign'])
plt.show()

## 3. K-Means Clustering
K-Means mempartisi data menjadi sekumpulan grup ($K$). Jarak setiap titik data dievaluasi terhadap titik pusat klaster (centroid) terdekatnya.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

# Simulasi data pelanggan menjadi 4 kelompok
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

# Mengaplikasikan algoritma K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
y_kmeans = kmeans.fit_predict(X_blobs)
centroids = kmeans.cluster_centers_

# Visualisasi klaster dan centroid
plt.figure(figsize=(8, 5))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_kmeans, cmap='viridis', s=50, alpha=0.6, edgecolors='k')
plt.scatter(centroids[:, 0], centroids[:, 1], c='red', s=200, marker='X', label='Centroids')
plt.title('K-Means Clustering (K=4)')
plt.legend()
plt.show()

## 4. Hierarchical Clustering (Dendrogram)
Metode ini membangun pohon klaster dari bawah ke atas (*agglomerative*). Jarak ambang batas pemotongan (*cut*) akan menentukan berapa klaster yang terbentuk.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

# Kita ambil sampel kecil (40 titik) agar dendrogram tidak terlalu padat
X_subset = X_blobs[:40]

# Melakukan penggabungan klaster (Linkage) menggunakan metode 'ward'
Z = linkage(X_subset, method='ward')

plt.figure(figsize=(10, 6))
dendrogram(Z, leaf_rotation=90, leaf_font_size=10, color_threshold=5)
plt.title('Dendrogram Hierarchical Clustering')
plt.xlabel('Indeks Sampel')
plt.ylabel('Jarak (Distance)')
plt.axhline(y=5, color='r', linestyle='--', label='Garis Potong (Cut)')
plt.legend()
plt.show()

## 5. Model-Based Clustering (Gaussian Mixture Models)
K-Means sering kesulitan pada data yang berbentuk elips memanjang karena mengasumsikan klaster berbentuk lingkaran (*spherical*). GMM mengasumsikan data adalah gabungan beberapa distribusi normal yang memungkinkan bentuk elips.

In [ ]:
from sklearn.mixture import GaussianMixture

# Kita tarik/rentangkan data blob tadi agar berbentuk elips
transformation = [[0.6, -0.6], [-0.4, 0.8]]
X_stretched = np.dot(X_blobs, transformation)

# 1. Menggunakan K-Means pada data elips (Biasanya gagal/kurang tepat)
kmeans_stretch = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X_stretched)

# 2. Menggunakan GMM pada data elips (Lebih adaptif)
gmm = GaussianMixture(n_components=4, random_state=42)
gmm_labels = gmm.fit_predict(X_stretched)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(X_stretched[:, 0], X_stretched[:, 1], c=kmeans_stretch, cmap='viridis', s=30, alpha=0.8)
ax[0].set_title('Kegagalan K-Means pada Data Elips')

ax[1].scatter(X_stretched[:, 0], X_stretched[:, 1], c=gmm_labels, cmap='viridis', s=30, alpha=0.8)
ax[1].set_title('Keberhasilan GMM (Model-Based Clustering)')

plt.tight_layout()
plt.show()

---
*Catatan Tambahan: Pemahaman tentang PCA dan metode Clustering di bab ini sangat esensial untuk tahap eksplorasi (EDA), di mana pola data dapat menuntun Anda kepada analisis yang lebih terarah dan bermakna.*